# 第4周 · 线性回归与梯度下降（含数学工作坊 1）

**本周结束时你能做到**：用 X、y、模型、损失四要素描述一个监督学习问题；从零实现梯度下降并解释学习率的作用；说明为什么要做特征标准化。

本周是课程的分水岭：前三周在学“怎么让计算机听话”，从本周起学“怎么让计算机**从数据中学习**”。学习的核心机制——梯度下降——你将亲手实现，而且它和第 20 周训练大模型用的是同一个机制。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week04 as ck

---
## S1 · 数学工作坊 1：向量、矩阵与“导数即斜率”

不用紧张：今天的每一个数学概念都立刻用 NumPy 验证。它们其实都是第 2 周用过的东西，今天只是给它们正式的名字。

### 向量与点积

向量 = 一列数 = 一维 NumPy 数组。**点积**：对应位置相乘、再求和。

先用循环写一个手工版 `my_dot(a, b)`，再和 NumPy 对比：

In [ ]:
def my_dot(a, b):
    """点积：对应位置相乘再求和"""
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(my_dot(a, b), np.dot(a, b), a @ b)    # 三种写法，同一个数
ck.dot(my_dot)

点积马上就有用：**线性模型的一次预测就是一次点积**——每个特征乘一个权重，再加起来。

### 矩阵乘法与 shape 规则

矩阵 = 数字表格 = 二维数组。**本课程最重要的约定：每一行是一个样本，每一列是一个特征。** `X.shape == (n, d)` 读作“n 个样本、d 个特征”。

矩阵乘法 `A @ B`：结果的第 i 行第 j 列 = A 的第 i 行与 B 的第 j 列的点积。shape 规则（内侧维度相等、然后被“消掉”）：

```
(n, d) @ (d, 1) → (n, 1)        # 矩阵 @ 矩阵
(n, d) @ (d,)   → (n,)          # 矩阵 @ 向量（我们最常用）
```

In [ ]:
X = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])      # (3, 2)：3 个样本、2 个特征
w = np.array([10.0, 1.0])       # (2,)：每个特征一个权重

print((X @ w).shape, X @ w)     # (3,)：每个样本一个预测值
print(X[0] @ w)                 # 第一个样本的预测 = 一次点积

内侧维度对不上就会报错——这是接下来十六周你最常见的错误，现在亲眼看一次（这个单元格**应该**报错）：

In [ ]:
w_bad = np.array([1.0, 2.0, 3.0])    # (3,)：X 的内侧是 2，对不上
X @ w_bad

最后一行的 `mismatch` 就是“内侧维度对不上”。**万能排查法：把每个数组的 `.shape` 打印出来，对着规则核对。**

### 导数就是斜率

机器学习的“学习”由一个问题驱动：**参数往哪个方向动一点，损失会变小？**答案由导数给出。导数是函数在一点的斜率——放大到足够近，任何光滑曲线看起来都是直线。

不用背求导公式，可以用**数值差分**直接把斜率“量”出来：在 x 两侧各取一小步，割线的斜率就近似切线的斜率。写 `num_derivative(f, x, h=1e-5)`，返回 `(f(x+h) - f(x-h)) / (2h)`：

In [ ]:
def num_derivative(f, x, h=1e-5):
    """数值差分近似 f 在 x 处的导数（斜率）"""
    return (f(x + h) - f(x - h)) / (2 * h)

def square(x):
    return x ** 2

print(num_derivative(square, 3.0))    # 教科书答案：2x = 6
print(num_derivative(np.sin, 0.0))    # 教科书答案：cos(0) = 1
ck.numerical_derivative(num_derivative)

亲眼看一下“导数即斜率”：

In [ ]:
xs = np.linspace(-3, 3, 100)
x0 = 1.5
slope = num_derivative(square, x0)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(xs, square(xs), label="f(x) = x^2")
ax.plot(xs, square(x0) + slope * (xs - x0), "--", label=f"tangent at x={x0} (slope {slope:.1f})")
ax.scatter([x0], [square(x0)], color="k", zorder=3)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("The derivative is the slope of the tangent line")
ax.legend()
plt.show()

斜率为正 → 往左走函数值变小；斜率为负 → 往右走变小。**往斜率的反方向走，函数值就下降。**记住这句话——一小时后它就是梯度下降的全部。

---
## S2 · 监督学习的四要素

**机器学习**：不再手写规则，而是给出数据和“什么算好”的标准，让算法自己找规则。监督学习用四样东西描述：

| 要素 | 含义 | 今天的例子 |
| --- | --- | --- |
| 数据 X | 特征，shape (n, d) | 一个输入变量 x |
| 标签 y | 想预测的量，shape (n,) | 连续数值（所以叫“回归”） |
| 模型 | 从 X 到预测值的函数族 | 直线 y_hat = w·x + b |
| 损失 | 一个数，衡量预测差多远 | 均方误差 MSE |

学习 = 找到让损失最小的参数 (w, b)。

### 合成数据：自带标准答案的“标准品”

研究新方法时，先在**自己造的、答案已知的数据**上验证——这就是你实验室里的阳性对照。按 y = 3x + 2 + 噪声 造 200 个点（**写死随机种子**：可复现实验的第一步）：

In [ ]:
rng = np.random.default_rng(42)          # 随机种子固定为 42
X = rng.uniform(-2, 2, size=(200, 1))    # (n, d) = (200, 1)：200 个样本、1 个特征
y = 3 * X[:, 0] + 2 + rng.normal(0, 1, size=200)

true_w, true_b = 3.0, 2.0                # 只有造数据的人才知道的“标准答案”

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(X[:, 0], y, s=10, alpha=0.6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Synthetic data: y = 3x + 2 + noise")
plt.show()

### 损失：均方误差（MSE）

预测 y_hat 与真实 y 的差叫**残差**。MSE 就是残差平方的平均值：

```
MSE = mean((y_hat - y) ** 2)
```

平方让正负误差不相互抵消，也让大错误受到更重的惩罚。写出来：

In [ ]:
def mse_loss(y_hat, y):
    """均方误差：残差平方的平均"""
    return np.mean((y_hat - y) ** 2)

print(mse_loss(np.array([1.0, 2.0, 3.0]), np.array([1.0, 1.0, 5.0])))   # (0 + 1 + 4) / 3
ck.mse_loss(mse_loss)

### 损失曲面：每个 (w, b) 对应一个损失

参数只有两个，可以奢侈一把：把候选参数全试一遍，对网格上的每个 (w, b) 算一次 MSE，画等高线图：

In [ ]:
ws = np.linspace(-1, 7, 81)
bs = np.linspace(-2, 6, 81)
loss_grid = np.zeros((len(bs), len(ws)))
for i, b_ in enumerate(bs):
    for j, w_ in enumerate(ws):
        loss_grid[i, j] = mse_loss(w_ * X[:, 0] + b_, y)

fig, ax = plt.subplots(figsize=(5, 4))
cs = ax.contour(ws, bs, loss_grid, levels=20)
ax.clabel(cs, inline=True, fontsize=7)
ax.scatter([true_w], [true_b], marker="*", s=150, color="red", label="true (w, b)")
ax.set_xlabel("w")
ax.set_ylabel("b")
ax.set_title("MSE loss surface: one bowl, bottom near (3, 2)")
ax.legend()
plt.show()

一个碗，碗底就是最优参数。但有个问题：**真实模型有上百万个参数，这张图画不出来**。模型能做的，只是站在当前这一点，感知脚下的坡度——这就够了。

### 手算一步梯度下降

先把 b 固定在 2，只调 w。损失关于 w 的导数（用微积分可以推出来，第 6 周工作坊会推）：

```
dL/dw = mean(2 * (y_hat - y) * x)
```

梯度下降的更新规则，就是刚才那句话：**往斜率的反方向走一小步**——

```
w ← w - 学习率 × 导数
```

In [ ]:
w_now, b_now = 0.0, 2.0
lr = 0.1                                   # 学习率：一步走多远

y_hat = w_now * X[:, 0] + b_now
grad_w = np.mean(2 * (y_hat - y) * X[:, 0])
print("当前 w =", w_now, "  损失 =", round(mse_loss(y_hat, y), 3), "  导数 =", round(grad_w, 3))

w_new = w_now - lr * grad_w
print("更新后 w =", round(w_new, 3), "  损失 =", round(mse_loss(w_new * X[:, 0] + b_now, y), 3))

导数是负的（往右损失会降），所以 w 增大、损失确实下降了。

顺手做一次 sanity check：用 S1 的数值差分验证解析导数没写错（第 7 周的“梯度检查”就是这个思想）：

In [ ]:
print("解析导数:", round(grad_w, 5))
print("数值导数:", round(num_derivative(lambda w_: mse_loss(w_ * X[:, 0] + b_now, y), w_now), 5))

---
## S3 · 从零实现向量化梯度下降

把“一次调一个参数”升级成“全部参数一起更新”。流程固定为五步，**这也是你以后每个训练脚本的骨架**：数据 → 模型 → 损失 → 更新循环 → 看 loss 曲线，最后与标准答案对比。

模型的向量化写法（X 是 (n, d)，w 是 (d,)，对任意多特征都成立）：

```
y_hat = X @ w + b
```

两个参数的梯度（解析式，d 个特征时 grad_w 是 (d,) 向量）：

```
grad_w = 2/n * X.T @ (y_hat - y)
grad_b = 2/n * sum(y_hat - y)
```

> 对照其他教材时注意：d2l、吴恩达的课程把损失写成 $\tfrac12$MSE（吴恩达写作 $\tfrac{1}{2m}\sum$），求导后系数 2 正好被消掉，所以你会看到没有 2 的梯度式。两种写法只差一个常数倍——相当于学习率差 2 倍，学到的最优参数完全一样。

写出一步更新 `gd_step`：

In [ ]:
def gd_step(X, y, w, b, lr):
    """做一步梯度下降，返回更新后的 (w, b)"""
    n = len(y)
    y_hat = X @ w + b
    grad_w = 2 / n * X.T @ (y_hat - y)
    grad_b = 2 / n * np.sum(y_hat - y)
    return w - lr * grad_w, b - lr * grad_b

ck.gd_step(gd_step)

### 训练循环

从 w=0、b=0 出发，反复调用 `gd_step`，并记录每一步的损失：

In [ ]:
def train(X, y, lr, n_iters):
    w = np.zeros(X.shape[1])
    b = 0.0
    losses = []
    for _ in range(n_iters):
        losses.append(mse_loss(X @ w + b, y))
        w, b = gd_step(X, y, w, b, lr)
    return w, b, losses

w_fit, b_fit, losses = train(X, y, lr=0.1, n_iters=200)
print(f"学到的 w = {w_fit[0]:.3f}, b = {b_fit:.3f}（造数据用的是 w=3, b=2）")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses)
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss")
ax.set_title("Loss falls fast, then flattens at the noise floor")
plt.show()

loss 曲线是训练的心电图，以后每次训练都先看它。损失降到 1 附近就不再降——因为数据里混了标准差为 1 的噪声，**完美的模型也只能到这里**。学到的参数非常接近标准答案 (3, 2)。

### 学习率：梯度下降唯一要调的旋钮

太小 → 走得太慢；太大 → 一步迈过碗底、越荡越高（发散）。扫一遍亲眼看：

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for lr in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    _, _, losses_lr = train(X, y, lr=lr, n_iters=100)
    ax.plot(np.minimum(losses_lr, 1e4), label=f"lr={lr:g}")   # 发散的曲线截在 1e4，防止把别人压扁
ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss (clipped at 1e4)")
ax.set_title("Too small: crawls. Too large: diverges. lr=0.1 fits this problem")
ax.legend()
plt.show()

lr=1.0 的曲线一路向上：每一步都迈过头，离碗底越来越远。lr=1e-4 则在 100 步里几乎没动。**没有万能学习率，只有对当前问题合适的学习率**——作业里你会把它做成一次正式的受控实验。

### 与 sklearn 对比：阳性对照

`sklearn.linear_model.LinearRegression` 用最小二乘公式直接解出最优参数，正好当我们的“标准品”：

In [ ]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression().fit(X, y)
print(f"sklearn:   w = {sk_model.coef_[0]:.6f}, b = {sk_model.intercept_:.6f}")
print(f"我们的 GD: w = {w_fit[0]:.6f}, b = {b_fit:.6f}")
print(f"差距: {abs(sk_model.coef_[0] - w_fit[0]):.2e}, {abs(sk_model.intercept_ - b_fit):.2e}")
ck.fitted_params(w_fit, b_fit)

差距小于亿分之一——**我们从零写的二十行代码，和工业级库给出同一个答案**。

既然 sklearn 一行就能解，为什么还要学梯度下降？因为最小二乘的求解公式**只对线性回归存在**；从第 7 周起的每一个神经网络，都只能靠梯度下降训练。今天的 `gd_step` 循环，就是你未来所有训练代码的祖先。

---
## S4 · 真实数据：California Housing

1990 年美国人口普查的加州房价数据：20,640 个街区 × 8 个特征，目标是街区房价中位数（**单位：10 万美元**）。第一次运行会联网下载约 1 MB 并缓存到本机，之后离线可用。

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X_house, y_house = housing.data, housing.target
print(X_house.shape)
print(list(X_house.columns))
X_house.describe().round(2)

特征包括街区的收入中位数（MedInc，单位万美元）、房龄、平均房间数、人口、经纬度等。先看最有希望的特征——收入 vs 房价：

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(X_house["MedInc"], y_house, s=2, alpha=0.2)
ax.set_xlabel("Median income (tens of thousands of USD)")
ax.set_ylabel("Median house value (hundreds of thousands of USD)")
ax.set_title("Income predicts house value, but prices are capped at 5.0")
plt.show()

这张图里有两个真实数据的“怪相”，都值得写进实验记录：

- 顶部一条**水平带**：房价在 5.0（50 万美元）被**截断**了——再贵的街区也记成 5.0；
- `describe()` 里 `AveRooms` 最大值 141 间/户？那是住户极少的度假区街区的统计伪影。

**看到怪数据，先查数据的出处和说明，再怀疑自己的代码。**官方说明就在 `housing.DESCR` 里：AveRooms 的怪值它有交代（户数少、空房多的街区，比如度假区，“每户房间数”会大得出奇）；但 **5.0 的截断它一个字也没提**——这只能从数据本身看出来。画一张目标值的直方图：

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(y_house, bins=50)
ax.set_xlabel("Median house value (hundreds of thousands of USD)")
ax.set_ylabel("Number of districts")
ax.set_title("A lone spike at the right edge: the target is capped at 5.0")
plt.show()
print("最大值:", y_house.max(), "  取到最大值的街区数:", int((y_house == y_house.max()).sum()))

最右边那根孤零零的高柱就是截断：近千个街区的房价都被记成了上限 5.00001。**文档不会告诉你一切——先看数据。**

### 单特征回归：只用收入能预测多准？

划分训练/测试集，在训练集上拟合、在**测试集**上算 R²（R² = 1 完美预测，R² = 0 相当于只会猜均值）：

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_house, y_house, test_size=0.25, random_state=42)

single = LinearRegression().fit(X_train[["MedInc"]], y_train)
r2_single = single.score(X_test[["MedInc"]], y_test)
print(f"只用 MedInc 的测试 R² = {r2_single:.3f}")
ck.calhousing_r2_single(r2_single)

### 全部 8 个特征

In [ ]:
full = LinearRegression().fit(X_train, y_train)
r2_all = full.score(X_test, y_test)
print(f"全部 8 个特征的测试 R² = {r2_all:.3f}")
ck.calhousing_r2_all(r2_all)

0.47 → 0.59：更多特征确实有帮助，但离 1 还远——房价还取决于数据里根本没有的东西（学区、房屋状况、年代……）。**特征的上限决定模型的上限。**

### 把我们的梯度下降搬上真实数据：会翻车

In [ ]:
X_raw = X_train.to_numpy()
y_raw = y_train.to_numpy()

_, _, losses_diverge = train(X_raw, y_raw, lr=1e-6, n_iters=50)
print(f"lr=1e-6：50 步后损失 = {losses_diverge[-1]:.3g}（发散！）")

_, _, losses_tiny = train(X_raw, y_raw, lr=1e-7, n_iters=200)
print(f"lr=1e-7：200 步后损失 = {losses_tiny[-1]:.3f}（不发散，但离最优的 0.52 还很远）")

症结：8 个特征的尺度天差地别——MedInc 大约 0–15，Population 动辄上千。同一个学习率对有的参数太大（发散）、对有的又太小（不动）。损失曲面不再是圆碗，而是一条又窄又长的峡谷。

### 解药：标准化

每个特征减去自己的均值、除以自己的标准差——正是第 2 周的 z-score！所有特征就站上了同一尺度。**注意：均值和标准差只在训练集上计算**（为什么？第 5 周讲数据泄漏时揭晓）：

In [ ]:
mu = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)
X_std = (X_raw - mu) / sigma            # 广播：第 2 周学过

w_std, b_std, losses_std = train(X_std, y_raw, lr=0.1, n_iters=200)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses_tiny, label="raw features, lr=1e-7")
ax.plot(losses_std, label="standardized, lr=0.1")
ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss")
ax.set_title("Standardization turns the canyon back into a bowl")
ax.legend()
plt.show()

from sklearn.metrics import r2_score
X_test_std = (X_test.to_numpy() - mu) / sigma     # 测试集用训练集的 mu 和 sigma
print("标准化 + 我们的梯度下降，测试 R² =", round(r2_score(y_test, X_test_std @ w_std + b_std), 3))

标准化之后，我们的梯度下降与 sklearn 并肩（R² ≈ 0.59）。**从第 7 周起，标准化（或它的变体）是喂给神经网络的所有数据的标配。**

### 预告：模型更强 = 更好？

线性模型只能画直线。给特征加上平方、立方项（多项式特征），模型就能弯曲。只取 15 个训练点试试直线和 12 次曲线：

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_small, y_small = X[:15], y[:15]          # 只用合成数据的前 15 个点
xs_plot = np.linspace(-2, 2, 200).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(X_small[:, 0], y_small, color="k", zorder=3, label="15 training points")
for degree in [1, 12]:
    poly = PolynomialFeatures(degree=degree)
    m = LinearRegression().fit(poly.fit_transform(X_small), y_small)
    ax.plot(xs_plot[:, 0], m.predict(poly.transform(xs_plot)), label=f"degree {degree}")
ax.set_ylim(-8, 12)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Degree 12 hits every point but learns the noise: overfitting")
ax.legend()
plt.show()

12 次曲线几乎穿过每个训练点——训练损失更低，但它学到的是**噪声**，在新数据上会一塌糊涂。模型更强不一定更好，怎么诊断、怎么防？这是第 5 周的主题。

### 本周小结

- 向量与矩阵：点积、`(n, d) @ (d,) → (n,)`、每一行是一个样本；shape 对不上先 print
- 导数即斜率；数值差分能验证解析梯度（sanity check）
- 四要素：X、y、模型、损失；合成数据是自带答案的阳性对照
- 梯度下降：`参数 ← 参数 - lr × 梯度`；loss 曲线是训练的心电图
- 学习率太小太慢、太大发散；标准化把峡谷变回圆碗
- 真实数据有截断、有离群值：先查出处，再怀疑代码